In [34]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [35]:
df = spark.read.json("data/transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [36]:
df = spark.read.json("data/transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

df.show(10, truncate=False)

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywno

In [37]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss")
)

df.printSchema()
df.show(5, truncate=False)

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 5 rows



In [38]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)

store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [39]:
from pyspark.sql.functions import min as _min, max as _max

category_stats = (
    df.groupBy("category")
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(_min("amount"), 2).alias("min_PLN"),
        _round(_max("amount"), 2).alias("max_PLN"),
    )
    .orderBy("category")
)

category_stats.show()

+-----------+----------+-------+-------+
|   category|  suma_PLN|min_PLN|max_PLN|
+-----------+----------+-------+-------+
|elektronika|1520770.69|    9.0| 9999.0|
|    książki| 851382.08|    5.0|9107.25|
|     odzież| 849877.55|    5.0|9696.63|
|    żywność| 789514.43|    5.0|6916.92|
+-----------+----------+-------+-------+



In [40]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)

hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [41]:
hourly_clean = (
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)

hourly_clean.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [42]:
store_30min = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "store",
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od", "store")
)

store_30min.show(100, truncate=False)

+-------------------+-------------------+--------+---------+---------+
|od                 |do                 |store   |liczba_tx|suma_PLN |
+-------------------+-------------------+--------+---------+---------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|Gdańsk  |252      |93391.22 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Kraków  |289      |117786.42|
|2026-04-12 08:00:00|2026-04-12 08:30:00|Warszawa|275      |88441.58 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Wrocław |296      |111540.59|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Gdańsk  |514      |209187.85|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Kraków  |532      |223541.41|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Warszawa|490      |182435.06|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Wrocław |502      |215587.17|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Gdańsk  |619      |253364.95|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Kraków  |590      |224358.03|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Warszawa|584      |214573.66|
|2026-

In [43]:
from pyspark.sql.functions import desc

krakow_hourly = (
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy(desc("suma_PLN"))
)

krakow_hourly.show(truncate=False)

+-------------------+-------------------+---------+---------+
|od                 |do                 |liczba_tx|suma_PLN |
+-------------------+-------------------+---------+---------+
|2026-04-12 09:00:00|2026-04-12 10:00:00|1169     |483309.86|
|2026-04-12 08:00:00|2026-04-12 09:00:00|821      |341327.83|
|2026-04-12 10:00:00|2026-04-12 11:00:00|532      |201259.26|
+-------------------+-------------------+---------+---------+



In [44]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)

sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [45]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)

sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)

print(f"Tumbling (1h):         {tumbling_rows} okien")
print(f"Sliding  (1h / 30min): {sliding_rows} okien")

Tumbling (1h):         3 okien
Sliding  (1h / 30min): 7 okien


In [46]:
# Sliding daje więcej wierszy, ponieważ okna przesuwne nakładają się na siebie.
# Przy oknie 1h i kroku 30 minut część transakcji może zostać przypisana do dwóch okien.
# Tumbling dzieli czas na rozłączne przedziały, więc każda transakcja trafia tylko do jednego okna.

In [47]:
hourly_clean.filter(
    (col("od") == "2026-05-10 09:00:00") |
    (col("od").cast("string").contains("09:00:00"))
).show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
+-------------------+-------------------+---------+----------+



In [48]:
hourly_clean.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [49]:
# groupBy("store") agreguje wszystkie transakcje danego sklepu razem, bez podziału na czas.
# groupBy(window(...), "store") agreguje dane osobno dla każdego sklepu i każdego okna czasowego.
# Dzięki temu widzimy, jak wyniki zmieniają się w czasie.

In [50]:
# Transakcja z godziny 09:30 może należeć do dwóch okien sliding: 09:00–10:00 oraz 09:30–10:30, dlatego zawiera ją 2 okna.

In [51]:
#PRACA DOMOWA LAB2

In [52]:
gdansk_lowest_avg = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "srednia_PLN",
    )
    .orderBy("srednia_PLN")
)

gdansk_lowest_avg.show(truncate=False)

+-------------------+-------------------+---------+-----------+
|od                 |do                 |liczba_tx|srednia_PLN|
+-------------------+-------------------+---------+-----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|766      |395.01     |
|2026-04-12 10:00:00|2026-04-12 11:00:00|558      |412.92     |
|2026-04-12 09:00:00|2026-04-12 10:00:00|1174     |415.91     |
+-------------------+-------------------+---------+-----------+



In [53]:
#Najniższa średnia kwotę transakcji dla Gdańska to 395,01

In [54]:
category_09_0930 = (
    df.groupBy(window("timestamp", "30 minutes"), "category")
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "category",
        "liczba_tx",
    )
    .filter(
        (col("od").cast("string").contains("09:00:00")) &
        (col("do").cast("string").contains("09:30:00"))
    )
    .orderBy("category")
)

category_09_0930.show(truncate=False)

+-------------------+-------------------+-----------+---------+
|od                 |do                 |category   |liczba_tx|
+-------------------+-------------------+-----------+---------+
|2026-04-12 09:00:00|2026-04-12 09:30:00|elektronika|611      |
|2026-04-12 09:00:00|2026-04-12 09:30:00|książki    |622      |
|2026-04-12 09:00:00|2026-04-12 09:30:00|odzież     |605      |
|2026-04-12 09:00:00|2026-04-12 09:30:00|żywność    |567      |
+-------------------+-------------------+-----------+---------+



In [55]:
#Ilości transakcji per kategoria zostały przedstawione w powyższej tabelce

In [56]:
peak_15min = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
    )
    .orderBy(desc("liczba_tx"))
)

peak_15min.show(truncate=False)

+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
|2026-04-12 09:00:00|2026-04-12 09:15:00|1171     |
|2026-04-12 09:30:00|2026-04-12 09:45:00|1156     |
|2026-04-12 08:45:00|2026-04-12 09:00:00|1139     |
|2026-04-12 09:45:00|2026-04-12 10:00:00|1100     |
|2026-04-12 08:30:00|2026-04-12 08:45:00|899      |
|2026-04-12 10:00:00|2026-04-12 10:15:00|858      |
|2026-04-12 08:15:00|2026-04-12 08:30:00|644      |
|2026-04-12 10:15:00|2026-04-12 10:30:00|582      |
|2026-04-12 08:00:00|2026-04-12 08:15:00|468      |
|2026-04-12 10:30:00|2026-04-12 10:45:00|443      |
|2026-04-12 10:45:00|2026-04-12 11:00:00|306      |
+-------------------+-------------------+---------+



In [57]:
#Najwięcej transakcji było w oknie czasowym od godziny 9:15 do godziny 9:30 i ta liczba wynosiła 1234

In [58]:
spark.stop()